Cell 1 — Imports + FAISS check (HNSW)

In [ ]:
import os
import time
import numpy as np
import pandas as pd

from typing import Tuple, List

import faiss
print("FAISS version:", faiss.__version__)

✅ FAISS version: 1.13.2


Cell 2 — Config (files + metrics + HNSW params + output)

In [2]:
# ---- Input embedding files ----
emb_folder = "E:\\Experiments\\Similarity Serach\\Embeddings"
EMBEDDING_FILES = {
    "bert_finetuned": emb_folder + "\\bert_finetuned_embeddings.xlsx",
    "gemini": emb_folder + "\\Gemini_Embedding.xlsx",
    "qwen3_8b": emb_folder + "\\Qwen3_Embedding_8B.xlsx",
    "sbert": emb_folder + "\\SBERT_Embedding_2_classification.xlsx",
}

TOPK_LIST = [1, 5, 10, 20]

# same similarity configs
SIM_METRICS = ["cosine", "dot", "L2"]

# ---- HNSW parameters ----
# number of neighbors in the graph (higher => better recall, more memory)
HNSW_M = 32

# construction quality (higher => better graph, slower build)
HNSW_EF_CONSTRUCTION = 200

# search breadth (higher => better recall, slower search)
HNSW_EF_SEARCH = 64

OUT_XLSX = "faiss_hnsw_results.xlsx"

Cell 3 — Loader

In [3]:
def is_numeric_col_name(c) -> bool:
    if isinstance(c, (int, np.integer)):
        return True
    s = str(c)
    return s.isdigit()

def load_embedding_xlsx(path: str) -> Tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    df = pd.read_excel(path, engine="openpyxl")

    # label column
    label_candidates = [c for c in df.columns if str(c).lower() in ("label", "y", "class")]
    if not label_candidates:
        raise ValueError(f"[{path}] No label column found.")
    label_col = label_candidates[0]

    # id column
    id_candidates = [c for c in df.columns if str(c).lower() in ("filename", "file", "text_file", "id", "file_id")]
    if id_candidates:
        preferred = [c for c in id_candidates if str(c).lower() in ("filename", "file", "text_file")]
        id_col = preferred[0] if preferred else id_candidates[0]
    else:
        non_num = [c for c in df.columns if c != label_col and not is_numeric_col_name(c)]
        if not non_num:
            raise ValueError(f"[{path}] No id-like column found.")
        id_col = non_num[0]

    # embedding columns
    emb_cols = [c for c in df.columns if c != label_col and is_numeric_col_name(c)]
    if not emb_cols:
        emb_cols = [c for c in df.columns
                    if c != label_col and str(c).lower().startswith("e") and str(c)[1:].isdigit()]
    if not emb_cols:
        raise ValueError(f"[{path}] No embedding columns found.")

    X = df[emb_cols].to_numpy(dtype=np.float32)
    y = df[label_col].to_numpy()
    ids = df[id_col].astype(str).to_numpy()

    return df, X, y, ids

Cell 4 — IR metrics

In [4]:
def precision_at_k(rels: np.ndarray, k: int) -> float:
    return float(np.sum(rels[:k])) / float(k)

def recall_at_k(rels: np.ndarray, k: int, total_relevant: int) -> float:
    if total_relevant <= 0:
        return 0.0
    return float(np.sum(rels[:k])) / float(total_relevant)

def dcg_at_k(rels: np.ndarray, k: int) -> float:
    rels_k = rels[:k]
    denom = np.log2(np.arange(2, k + 2))
    return float(np.sum(rels_k / denom))

def ndcg_at_k(rels: np.ndarray, k: int) -> float:
    dcg = dcg_at_k(rels, k)
    ideal = np.sort(rels)[::-1]
    idcg = dcg_at_k(ideal, k)
    return 0.0 if idcg == 0 else (dcg / idcg)

def mrr_at_k(rels: np.ndarray, k: int) -> float:
    rels_k = rels[:k]
    idx = np.where(rels_k == 1)[0]
    return 0.0 if len(idx) == 0 else (1.0 / float(idx[0] + 1))

Cell 5 — HNSW Retrieval (L2 / dot / cosine) + PerQuery output

In [5]:
from sklearn.preprocessing import normalize

def faiss_hnsw_eval_all3(
    X: np.ndarray, y: np.ndarray, ids: np.ndarray,
    topk_list: List[int],
    M: int,
    ef_construction: int,
    ef_search: int
) -> pd.DataFrame:
    """
    Graph-based ANN with FAISS HNSW:
      - IndexHNSWFlat(L2) -> metric='L2'
      - IndexHNSWFlat(IP) -> metric='dot'
      - IndexHNSWFlat(IP) on normalized -> metric='cosine'
    Relevance: same label.
    """
    X = np.asarray(X, dtype=np.float32)
    N, d = X.shape

    maxK = max(topk_list)
    search_k = min(N, maxK + 1)  # +1 for self

    rows = []

    def eval_one(metric_name: str, X_use: np.ndarray, faiss_metric: str):
        if faiss_metric == "L2":
            index = faiss.IndexHNSWFlat(d, M, faiss.METRIC_L2)
        elif faiss_metric == "IP":
            index = faiss.IndexHNSWFlat(d, M, faiss.METRIC_INNER_PRODUCT)
        else:
            raise ValueError("faiss_metric must be 'L2' or 'IP'")

        # set HNSW params
        index.hnsw.efConstruction = int(ef_construction)
        index.hnsw.efSearch = int(ef_search)

        index.add(X_use)

        # batch search
        D, I = index.search(X_use, search_k)

        for i in range(N):
            nbrs = I[i]
            nbrs = nbrs[nbrs != i]  # remove self
            if len(nbrs) == 0:
                continue
            nbrs = nbrs[:maxK]

            rel = (y[nbrs] == y[i]).astype(np.int32)
            total_rel = int(np.sum(y == y[i]) - 1)

            for K in topk_list:
                Ke = min(K, len(rel))
                rows.append({
                    "method": f"faiss_hnsw_m{M}_efc{ef_construction}_efs{ef_search}",
                    "metric": metric_name,
                    "query_id": ids[i],
                    "query_label": int(y[i]),
                    "K": int(K),
                    "precision@K": precision_at_k(rel, Ke),
                    "recall@K": recall_at_k(rel, Ke, total_rel),
                    "ndcg@K": ndcg_at_k(rel, Ke),
                    "mrr@K": mrr_at_k(rel, Ke),
                })

    # 1) L2
    eval_one("L2", X, "L2")

    # 2) dot
    eval_one("dot", X, "IP")

    # 3) cosine
    Xn = normalize(X, axis=1).astype(np.float32)
    eval_one("cosine", Xn, "IP")

    return pd.DataFrame(rows)

Cell 6 — Test HNSW on SBERT (Sanity check)

In [6]:
emb_name = "sbert"
path = EMBEDDING_FILES[emb_name]
df0, X0, y0, ids0 = load_embedding_xlsx(path)

t0 = time.time()
perq_sbert_hnsw = faiss_hnsw_eval_all3(
    X0, y0, ids0,
    topk_list=TOPK_LIST,
    M=HNSW_M,
    ef_construction=HNSW_EF_CONSTRUCTION,
    ef_search=HNSW_EF_SEARCH
)
print("Done HNSW:", emb_name, "| rows:", len(perq_sbert_hnsw), "| seconds:", round(time.time()-t0, 2))

print(perq_sbert_hnsw["metric"].value_counts())
perq_sbert_hnsw.head()

Done HNSW: sbert | rows: 27876 | seconds: 1.03
metric
L2        9292
dot       9292
cosine    9292
Name: count, dtype: int64


,method,metric,query_id,query_label,K,precision@K,recall@K,ndcg@K,mrr@K
0,faiss_hnsw_m32_efc200_efs64,L2,1000_78_shopping__Ni_I4Fk5Eem2zdxW8Rsq_g.txt,78,1,1.0,0.007463,1.0,1.0
1,faiss_hnsw_m32_efc200_efs64,L2,1000_78_shopping__Ni_I4Fk5Eem2zdxW8Rsq_g.txt,78,5,1.0,0.037313,1.0,1.0
2,faiss_hnsw_m32_efc200_efs64,L2,1000_78_shopping__Ni_I4Fk5Eem2zdxW8Rsq_g.txt,78,10,1.0,0.074627,1.0,1.0
3,faiss_hnsw_m32_efc200_efs64,L2,1000_78_shopping__Ni_I4Fk5Eem2zdxW8Rsq_g.txt,78,20,1.0,0.149254,1.0,1.0
4,faiss_hnsw_m32_efc200_efs64,L2,1001_18_computer__wJS0wCqtEeiGfabwoJ7AXg.txt,18,1,1.0,0.006452,1.0,1.0


Cell 7 — Run ALL embeddings + Summary + Save Excel (HNSW)

In [ ]:
def summarize(perquery: pd.DataFrame, embedding_name: str) -> pd.DataFrame:
    grp = perquery.groupby(["method", "metric", "K"], as_index=False).agg({
        "precision@K": "mean",
        "recall@K": "mean",
        "ndcg@K": "mean",
        "mrr@K": "mean",
    })
    grp.insert(0, "embedding", embedding_name)
    return grp

all_perquery = []
all_summary = []

for emb_name, path in EMBEDDING_FILES.items():
    print(f"\n=== FAISS HNSW: {emb_name} ===")
    if not os.path.exists(path):
        print("File not found:", path)
        continue

    df_e, X, y, ids = load_embedding_xlsx(path)

    t0 = time.time()
    perq = faiss_hnsw_eval_all3(
        X, y, ids,
        topk_list=TOPK_LIST,
        M=HNSW_M,
        ef_construction=HNSW_EF_CONSTRUCTION,
        ef_search=HNSW_EF_SEARCH
    )
    perq.insert(0, "embedding", emb_name)
    all_perquery.append(perq)

    summ = summarize(perq, emb_name)
    all_summary.append(summ)

    print("perquery rows:", len(perq), "| seconds:", round(time.time()-t0, 2))

df_perquery = pd.concat(all_perquery, ignore_index=True)
df_summary  = pd.concat(all_summary, ignore_index=True)

with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as w:
    df_summary.to_excel(w, sheet_name="Summary", index=False)
    df_perquery.to_excel(w, sheet_name="PerQuery", index=False)

print("\nSaved:", OUT_XLSX)
df_summary.head(20)


=== FAISS HNSW: bert_finetuned ===
  ✅ perquery rows: 27876 | seconds: 1.9

=== FAISS HNSW: gemini ===
  ✅ perquery rows: 27876 | seconds: 1.08

=== FAISS HNSW: qwen3_8b ===
  ✅ perquery rows: 27876 | seconds: 3.27

=== FAISS HNSW: sbert ===
  ✅ perquery rows: 27876 | seconds: 0.81

✅ Saved: faiss_hnsw_results.xlsx


,embedding,method,metric,K,precision@K,recall@K,ndcg@K,mrr@K
0,bert_finetuned,faiss_hnsw_m32_efc200_efs64,L2,1,0.950495,0.013989,0.950495,0.950495
1,bert_finetuned,faiss_hnsw_m32_efc200_efs64,L2,5,0.934998,0.068434,0.944683,0.960877
2,bert_finetuned,faiss_hnsw_m32_efc200_efs64,L2,10,0.930865,0.135856,0.947014,0.961511
3,bert_finetuned,faiss_hnsw_m32_efc200_efs64,L2,20,0.925398,0.269264,0.964032,0.961814
4,bert_finetuned,faiss_hnsw_m32_efc200_efs64,cosine,1,0.950495,0.013989,0.950495,0.950495
5,bert_finetuned,faiss_hnsw_m32_efc200_efs64,cosine,5,0.935084,0.068437,0.944803,0.960827
6,bert_finetuned,faiss_hnsw_m32_efc200_efs64,cosine,10,0.930951,0.135859,0.947094,0.961478
7,bert_finetuned,faiss_hnsw_m32_efc200_efs64,cosine,20,0.925743,0.269358,0.964099,0.961820
8,bert_finetuned,faiss_hnsw_m32_efc200_efs64,dot,1,0.951356,0.014022,0.951356,0.951356
9,bert_finetuned,faiss_hnsw_m32_efc200_efs64,dot,5,0.935342,0.068467,0.945453,0.961279
